In [1]:
import pandas as pd

# ---------------------------------------------------------
# 1. 데이터 로드 (6개 파일)
# ---------------------------------------------------------
print("1. 데이터 로딩 중... (시간이 조금 걸릴 수 있습니다)")
orders = pd.read_csv('orders.csv')
prior = pd.read_csv('order_products__prior.csv')
train = pd.read_csv('order_products__train.csv')
products = pd.read_csv('products.csv')
departments = pd.read_csv('departments.csv')
aisles = pd.read_csv('aisles.csv')

# ---------------------------------------------------------
# 2. Orders 정리 (Test 삭제 & eval_set 컬럼 삭제)
# ---------------------------------------------------------
print("2. Orders 테이블 정리 중...")

# (1) Test 데이터 행(Row) 삭제
orders = orders[orders['eval_set'] != 'test']

# (2) eval_set 컬럼 삭제 (요청사항 1번)
# 이미 prior/train만 남았고, order_number로 최신 여부를 알 수 있으므로 삭제합니다.
orders.drop(columns=['eval_set'], inplace=True)

# ---------------------------------------------------------
# 3. 상품 구매 내역 합치기 (Prior + Train)
# ---------------------------------------------------------
print("3. 구매 내역 위아래로 합치는 중...")
all_order_products = pd.concat([prior, train], axis=0)

# 메모리 확보를 위해 사용한 변수 삭제
del prior, train

# ---------------------------------------------------------
# 4. 데이터 병합 (Merge)
# ---------------------------------------------------------
print("4. 전체 데이터 병합 및 중복 ID 제거 중...")

# (1) 주문 정보(User, 시간 등) 붙이기
# on='order_id' 기준, inner join으로 test 주문은 자연스럽게 제외됨
merged_df = pd.merge(all_order_products, orders, on='order_id', how='inner')

# (2) 상품 상세 정보 붙이기
merged_df = pd.merge(merged_df, products, on='product_id', how='left')

# (3) 카테고리(Department) 붙이기 & 중복 ID 삭제
merged_df = pd.merge(merged_df, departments, on='department_id', how='left')
merged_df.drop(columns=['department_id'], inplace=True)

# (4) 세부 카테고리(Aisle) 붙이기 & 중복 ID 삭제
merged_df = pd.merge(merged_df, aisles, on='aisle_id', how='left')
merged_df.drop(columns=['aisle_id'], inplace=True)

# # product_id 날리기
# merged_df.drop(columns=['product_id'], inplace=True)

# ---------------------------------------------------------
# 5. 파일 내보내기 (요청사항 2번)
# ---------------------------------------------------------
print("-" * 30)
print(f"최종 데이터 크기: {merged_df.shape}")
print("최종 컬럼 목록:", merged_df.columns.tolist())
print("-" * 30)

# print("5. train.csv 파일로 저장 중... (용량이 크니 기다려주세요)")
# # index=False로 해야 불필요한 인덱스 숫자가 안 생깁니다.
# merged_df.to_csv('train.csv', index=False)

# print("✅ 저장 완료! 'train.csv' 파일이 생성되었습니다.")

1. 데이터 로딩 중... (시간이 조금 걸릴 수 있습니다)
2. Orders 테이블 정리 중...
3. 구매 내역 위아래로 합치는 중...
4. 전체 데이터 병합 및 중복 ID 제거 중...
------------------------------
최종 데이터 크기: (33819106, 12)
최종 컬럼 목록: ['order_id', 'product_id', 'add_to_cart_order', 'reordered', 'user_id', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order', 'product_name', 'department', 'aisle']
------------------------------


In [2]:
merged_df.head(20)

,order_id,add_to_cart_order,reordered,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_name,department,aisle
0,2,1,1,202279,3,5,9,8.0,Organic Egg Whites,dairy eggs,eggs
1,2,2,1,202279,3,5,9,8.0,Michigan Organic Kale,produce,fresh vegetables
2,2,3,0,202279,3,5,9,8.0,Garlic Powder,pantry,spices seasonings
3,2,4,1,202279,3,5,9,8.0,Coconut Butter,pantry,oils vinegars
4,2,5,0,202279,3,5,9,8.0,Natural Sweetener,pantry,baking ingredients
5,2,6,1,202279,3,5,9,8.0,Carrots,produce,fresh vegetables
6,2,7,1,202279,3,5,9,8.0,Original Unflavored Gelatine Mix,pantry,doughs gelatins bake mixes
7,2,8,1,202279,3,5,9,8.0,All Natural No Stir Creamy Almond Butter,pantry,spreads
8,2,9,0,202279,3,5,9,8.0,Classic Blend Cole Slaw,produce,packaged vegetables fruits
9,3,1,1,205970,16,5,17,12.0,Total 2% with Strawberry Lowfat Greek Strained...,dairy eggs,yogurt


In [2]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33819106 entries, 0 to 33819105
Data columns (total 12 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   order_id                int64  
 1   product_id              int64  
 2   add_to_cart_order       int64  
 3   reordered               int64  
 4   user_id                 int64  
 5   order_number            int64  
 6   order_dow               int64  
 7   order_hour_of_day       int64  
 8   days_since_prior_order  float64
 9   product_name            object 
 10  department              object 
 11  aisle                   object 
dtypes: float64(1), int64(8), object(3)
memory usage: 3.0+ GB


In [4]:
import pandas as pd
import numpy as np

def reduce_mem_usage(df):
    """ 데이터프레임의 메모리 사용량을 줄이는 함수 """
    start_mem = df.memory_usage().sum() / 1024**2
    print(f'기존 메모리 사용량: {start_mem:.2f} MB')

    for col in df.columns:
        col_type = df[col].dtype

        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()
            
            # 정수형(Integer) 줄이기
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            # 실수형(Float) 줄이기
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32) # float16은 정확도 이슈로 32 추천
        else:
            # 문자열(Object) -> 카테고리(Category)
            df[col] = df[col].astype('category')

    end_mem = df.memory_usage().sum() / 1024**2
    print(f'최적화 후 메모리 사용량: {end_mem:.2f} MB')
    print(f'감소율: {100 * (start_mem - end_mem) / start_mem:.1f}%')
    return df

# 사용법
merged_df = reduce_mem_usage(merged_df)

# # 그 다음 저장 (pickle 형식이 csv보다 훨씬 빠르고 용량도 작음)
# merged_df.to_pickle('instacart_optimized.pkl') 
# # 불러올 땐: pd.read_pickle('instacart_optimized.pkl')

기존 메모리 사용량: 2838.21 MB
최적화 후 메모리 사용량: 807.70 MB
감소율: 71.5%


In [5]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33819106 entries, 0 to 33819105
Data columns (total 11 columns):
 #   Column                  Dtype   
---  ------                  -----   
 0   order_id                int32   
 1   add_to_cart_order       int16   
 2   reordered               int8    
 3   user_id                 int32   
 4   order_number            int8    
 5   order_dow               int8    
 6   order_hour_of_day       int8    
 7   days_since_prior_order  float32 
 8   product_name            category
 9   department              category
 10  aisle                   category
dtypes: category(3), float32(1), int16(1), int32(2), int8(4)
memory usage: 807.7 MB


In [ ]:
# pip3.12 install pyarrow --break-system-packages

In [6]:
import pandas as pd

# 가정: merged_df가 이미 준비되어 있고, 메모리 최적화(category 변환 등)가 끝난 상태
# index=False: 불필요한 인덱스 번호 저장 안 함 (용량 절약)
merged_df.to_parquet('instacart_full_data.parquet', engine='pyarrow', index=False)

print("Parquet 파일 저장 완료!")

Parquet 파일 저장 완료!


In [7]:
import pandas as pd

# 엔진을 명시해주면 더 안정적입니다.
df = pd.read_parquet('instacart_full_data.parquet', engine='pyarrow')

print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33819106 entries, 0 to 33819105
Data columns (total 11 columns):
 #   Column                  Dtype   
---  ------                  -----   
 0   order_id                int32   
 1   add_to_cart_order       int16   
 2   reordered               int8    
 3   user_id                 int32   
 4   order_number            int8    
 5   order_dow               int8    
 6   order_hour_of_day       int8    
 7   days_since_prior_order  float32 
 8   product_name            category
 9   department              category
 10  aisle                   category
dtypes: category(3), float32(1), int16(1), int32(2), int8(4)
memory usage: 807.7 MB
None


In [8]:
df.head(50)

,order_id,add_to_cart_order,reordered,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_name,department,aisle
0,2,1,1,202279,3,5,9,8.0,Organic Egg Whites,dairy eggs,eggs
1,2,2,1,202279,3,5,9,8.0,Michigan Organic Kale,produce,fresh vegetables
2,2,3,0,202279,3,5,9,8.0,Garlic Powder,pantry,spices seasonings
3,2,4,1,202279,3,5,9,8.0,Coconut Butter,pantry,oils vinegars
4,2,5,0,202279,3,5,9,8.0,Natural Sweetener,pantry,baking ingredients
5,2,6,1,202279,3,5,9,8.0,Carrots,produce,fresh vegetables
6,2,7,1,202279,3,5,9,8.0,Original Unflavored Gelatine Mix,pantry,doughs gelatins bake mixes
7,2,8,1,202279,3,5,9,8.0,All Natural No Stir Creamy Almond Butter,pantry,spreads
8,2,9,0,202279,3,5,9,8.0,Classic Blend Cole Slaw,produce,packaged vegetables fruits
9,3,1,1,205970,16,5,17,12.0,Total 2% with Strawberry Lowfat Greek Strained...,dairy eggs,yogurt


In [9]:
df.tail(50)

,order_id,add_to_cart_order,reordered,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_name,department,aisle
33819056,3420998,11,1,123299,25,6,18,30.0,Organic Cilantro,produce,fresh herbs
33819057,3420998,12,1,123299,25,6,18,30.0,Monterey Jack Cheese,dairy eggs,packaged cheese
33819058,3420998,13,0,123299,25,6,18,30.0,Fusilli No. 34,dry goods pasta,dry pasta
33819059,3420998,14,0,123299,25,6,18,30.0,Pesto Alla Genovese Basil,dry goods pasta,pasta sauce
33819060,3420998,15,0,123299,25,6,18,30.0,Penne Rigate #41 Pasta,dry goods pasta,dry pasta
33819061,3420998,16,1,123299,25,6,18,30.0,Sprouted Multi-Grain Bread,bakery,bread
33819062,3420998,17,0,123299,25,6,18,30.0,Spaghetti No 12,dry goods pasta,dry pasta
33819063,3420998,18,1,123299,25,6,18,30.0,Organic Balsamic Vinegar Of Modena,pantry,oils vinegars
33819064,3420998,19,1,123299,25,6,18,30.0,Sheep Milk Vanilla Yogurt,dairy eggs,yogurt
33819065,3420998,20,1,123299,25,6,18,30.0,Apple Honeycrisp Organic,produce,fresh fruits
